# AI-Assisted Replication and Red-Team of a Peer's Package

**Topic 13 · async module**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb13_replication_redteam_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** all positions (framing / diagnosis / communication). This is
a **self-paced async module**: no live meeting, everything you need is in this
notebook, and it fits one focused sitting of about 50 minutes. You take a peer's
finished research package, rebuild its number, and audit it. The compass position
is whatever your assigned peer declared. Your job is not to make a new claim of
your own. Your job is to test whether theirs holds.

**Design pathway:** cross-cutting (reproduction). **Reproduction** means
regenerating a study's reported number from the package it shipped, using nothing
outside that package. Example: you open a classmate's notebook, run it top to
bottom, and check whether the same turnout figure comes back out. Reproduction
tests whether a result *regenerates*, which is a different question from whether it
is *true*.

| | |
|---|---|
| **A claim this topic PERMITS** | "I reproduced the headline number from the package alone, and here are the three weaknesses I ranked by how much each one threatens the claim." |
| **A claim this topic does NOT permit** | "It ran, so it's fine." A clean run is not agreement between the write-up's claims and what the computation actually shows. |

**Where this sits in the course:** the Thanksgiving-week async module. It develops
**M13 — Replication and Red-Team Report** on your assigned peer package, due
Sunday night. The audit skill you build here is the one you will want turned on
your own dossier when class resumes.

*Provenance: project/final_dossier/reproducibility_audit_protocol.md + the GenAI Studio 'Reproducibility Auditor' role touchpoint (M13) | cold-reproduction + claim-to-output trace + red-team protocol | self-contained reproduce-and-audit module on a real course dataset (foos_etal) with a required GenAI Studio pass | newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why a clean **restart-and-run-all** is evidence a package *runs*, not
   evidence its claims are *true*.
2. Trace a package's **data and code lineage** and reproduce its **headline
   number** from the package alone.
3. Test **claims-vs-computation agreement**: line up each written claim against
   what the code actually produces, and flag every gap.
4. Probe an **alternative specification** and a **hidden assumption**, and show how
   each one moves or weakens the headline.
5. Name the **AI-related weaknesses** a package can hide behind an **illusion of
   completeness**, and verify a GenAI Studio audit pass yourself instead of
   trusting it.
6. Turn your findings into **prioritized recommendations** ranked by threat to the
   claim, and apply the whole audit to your own M13 peer package.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the lineage diagram; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)  # used for the street-cluster resampling below

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This module leans on **Gemini** and on Purdue's **GenAI Studio Reproducibility
Auditor**, a course-configured role that reads a package the way a cold replicator
would. Both are strong at one job: reading a stranger's package fast and listing
where it *might* break. Neither can run the code, and neither can decide whether a
claim survives. A **reproducibility package** is the full bundle a study ships so
someone else can rebuild its numbers: the data, the code, the run order, the
seeds, and the write-up. You will point AI at that bundle to generate a list of
suspects, then you run the code and confirm every suspect yourself.

**Never delegate these:** whether the headline number actually reproduced (only
your run settles that), which weakness most threatens the claim, and how you rank
your recommendations. The full list of never-delegate decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own answer
before you open the tool, so its output has something of yours to disagree with.

> **A question that often comes up here:** *"If the notebook runs top to bottom
> with no errors, hasn't it already passed?"* No, and this is the whole module.
> A clean run proves the code *executes*. It says nothing about whether the
> write-up's claims match what the code computed, whether a different defensible
> choice would move the number, or whether an assumption buried in the analysis is
> false. Those are the questions a red-team asks, and none of them is answered by
> a green check.

---

## 1. The Package on Your Desk

**Guiding question:** *what did you actually receive, and what does rebuilding its
number prove?*

> *"Anyone can put a number on a poster. My question is whether a stranger, with
> only the files you handed them, gets that same number back. If they cannot, I do
> not care how good the poster looked."*
> — a **journal reviewer**, describing the standard your peer's package now has to meet

Your assigned package is a classmate's finished study, anonymized so you cannot
tell whose it is. Two words describe what you are about to test.

- **Headline number**: the single figure the study's main claim rests on. Example:
  "canvassing raised turnout by 4 points." The whole poster stands or falls on
  that one number, so it is the first thing you rebuild.
- **Restart-and-run-all**: clearing the notebook's memory and running every cell
  from the top in order, with no manual fixes. Example: in Colab, "Restart runtime"
  then "Run all." If a number only appears when cells are run out of order, it is
  not reproducible.

To practice on real numbers before you open your assigned package, this module
ships a **sample package** built on a genuine course dataset: a UK
get-out-the-vote field experiment where some streets were canvassed door to door
and some were not. You will treat it exactly as you would a peer's package. Rebuild
its headline number first, then audit it.

> **A question that often comes up here:** *"Isn't 'reproduction' just the same
> word as 'replication'?"* They point at different things, and the difference
> matters for M13. **Reproduction** is getting the same number from the *same*
> data and code the study shipped. **Replication** is getting a similar result
> from a *new* study or new data. This module is reproduction: same package, same
> number, run by a different person. You are testing the package, not re-running
> the world.

## 2. Cold Reproduction: Rebuild the Headline Number

**Guiding question:** *can you get the package's number back out, using only what
is inside the package?*

Before any number, trace the **data and code lineage**: the chain from raw data,
through each processing step, to the final figure the claim quotes. Example: a
turnout file, filtered to one election, grouped by whether a street was canvassed,
differenced, and printed as "4 points." When a claim floats free of that chain,
with no cell that actually produces it, you have found your first problem. Draw the
chain before you trust the number.

**Deeper pass — run this on your own during the module.**
**Before you ask:** in one sentence, name the single link in a data-to-claim chain
where you think a number is most likely to detach from the code. Commit your guess
before the tool offers its list.

> 💡 **Gemini Prompt:** "I am cold-reproducing a peer's research package. Here is
> everything it shipped: [paste the package's README, its data description, its
> code, and its written headline claim]. As a reproducibility auditor, list every
> input or step a stranger needs and might not find: the data source and version,
> the run order, any seed, and any hard-coded path. Put it in a table with columns
> for the item, whether the package supplies it, and the exact cell where I should
> check. Then, as a hostile reviewer, name the one link where the written claim is
> most likely to have detached from an actual output."
>
> **After running, verify (counters *illusion of completeness*: a long, tidy gap
> list can still omit the one missing pointer that blocks the whole run):**
> - [ ] Check the list against the link you committed to above. If yours is
>   missing, the tidy table just failed the one test it looked complete on.
> - [ ] Map each item it names to a real cell in the package. An item that points
>   to no cell is one the tool invented, not one the package is missing.
> - [ ] Confirm it did not silently assume a step "works." Only your own run
>   settles that, so mark every unverified step to test yourself.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Draw the data-and-code lineage: raw data -> processing -> headline number -> poster claim.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Raw data\n(canvass file)": (0.0, 1.0),
    "Processing\n(filter, group)": (1.6, 1.0),
    "Headline number\n(the computed diff)": (3.2, 1.0),
    "Poster claim\n(the written sentence)": (4.8, 1.0),
    "Undisclosed\nchoices": (1.6, 2.15),
}
edges = [
    ("Raw data\n(canvass file)", "Processing\n(filter, group)"),
    ("Processing\n(filter, group)", "Headline number\n(the computed diff)"),
    ("Headline number\n(the computed diff)", "Poster claim\n(the written sentence)"),
    ("Undisclosed\nchoices", "Processing\n(filter, group)"),
]
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(10, 4.5))
nx.draw_networkx_nodes(G, pos, node_color="#DCE6F1", edgecolors="#4C72B0",
                       linewidths=1.5, node_size=6200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8.5, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=18, edge_color="#555555",
                       width=1.6, node_size=6200, ax=ax)
edge_labels = {
    ("Headline number\n(the computed diff)", "Poster claim\n(the written sentence)"):
        "the link a red-team audits",
}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8,
                             label_pos=0.5, ax=ax)
ax.set_title("Data-and-code lineage: where a written claim can detach from the computation")
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Lineage drawn.")
print("  Reproduction rebuilds the middle of the chain: data -> processing -> number.")
print("  The red-team audits the last arrow: does the written claim match that number?")
print("  The top arrow is the quiet danger: choices made in processing that the")
print("  write-up never mentions can move the number without anyone noticing.")

**Reading the diagram.** Follow the chain left to right. Raw data becomes a
processed table, the table produces one number, and that number becomes a sentence
on a poster. Reproduction rebuilds the middle links and checks that the number
comes back. The red-team stares at the final arrow, from number to sentence, and
at the top arrow, the **undisclosed choices** feeding processing, because that is
where a defensible-looking claim quietly parts ways with its evidence.

*In plain terms, the picture says a poster claim is only as good as the unbroken
chain of code behind it, and two arrows are where chains break.*

### 🔮 Pause & Predict

You are about to run the sample package's own analysis: it loads the canvass file,
splits streets into canvassed and not-canvassed, and reports the turnout gap. The
poster claim you were handed reads: **"Our door-to-door canvassing campaign raised
turnout by 4 percentage points."** Before you run the next cell, commit: when you
rebuild the number from the code alone, will you get exactly 4 points, a bit more,
or a bit less? Write one sentence of reasoning. You are predicting a reveal, so do
not peek at the output first.

### YOUR ANSWER HERE:

**My predicted reproduced number (and how close to the poster's 4 points):**

**One sentence of reasoning:**

---

### 🛠️ Run the Study: cold-reproduce the headline number

Run the cell below. It is the sample package's analysis, run from a clean start:
load the data, keep the outcome and the canvass flag, and difference the two group
means. Read the reproduced number before you read the next markdown cell.

**🔵 Run this one during your module sitting.**
**Before you ask:** commit, in one sentence, to the number you expect before you
read Gemini's walkthrough (you set a direction last cell; now commit the value).

> 💡 **Gemini Prompt:** "Here is a Python cell that reproduces a study's headline
> number by differencing two group means: [paste the next cell]. Explain, line by
> line, what it computes, and confirm which printed value is the headline number a
> reader should compare against the poster. Then give me one independent way to
> confirm that number without rerunning this exact code, so I am not just trusting
> your walkthrough."
>
> **After running, verify (counters *confident fabrication*: a fluent line-by-line
> story can describe a number the code never printed):**
> - [ ] Confirm the headline value Gemini names matches the `diff_pts` your cell
>   actually printed, not a rounded number it guessed.
> - [ ] Run its "independent way to confirm" yourself. If it will not run, or lands
>   on a different value than your printout, trust your printout.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# THE SAMPLE PACKAGE'S ANALYSIS, reproduced cold from a clean start.
# The package ships a UK get-out-the-vote field experiment: `treat` = street was
# canvassed door-to-door; `marked_register_2014` = 1 if the person voted.
pkg = load_course_data("foos_etal.csv")
assert pkg.shape == (8375, 5), "unexpected shape — flag this as a lineage break!"
print("✓ Loaded the package data —", pkg.shape[0], "rows,", pkg.shape[1], "columns")
print("  Columns:", list(pkg.columns))

y = "marked_register_2014"
canvassed = pkg[pkg["treat"] == 1][y]      # streets that were canvassed
not_canv  = pkg[pkg["treat"] == 0][y]      # streets that were not
print(f"\n  canvassed  : {len(canvassed):>5} people, turnout {canvassed.mean():.4f}")
print(f"  not canvassed: {len(not_canv):>5} people, turnout {not_canv.mean():.4f}")

diff = canvassed.mean() - not_canv.mean()  # THE HEADLINE NUMBER (a difference in means)
diff_pts = diff * 100
print(f"\n  HEADLINE NUMBER (turnout gap) = {diff:.4f} = {diff_pts:.2f} percentage points")

In [ ]:
# Self-check: the reproduced number is close to, but not exactly, the poster's 4 points.
assert abs(diff_pts - 3.41) < 0.05, "reproduced gap drifted — investigate the lineage"
assert canvassed.mean() > not_canv.mean(), "canvassed streets should show higher turnout"
print(f"✓ Self-check passed: the package reproduces a {diff_pts:.2f}-point turnout gap.")
print(f"  The poster wrote '4 points.' The code produces {diff_pts:.2f}. Hold that difference.")

**Reading the evidence.** The package reproduces cleanly. Run from a clean
start, its code returns a turnout gap of **3.41 percentage points** (canvassed
streets voted at 50.66%, not-canvassed at 47.25%). That is a real success: the
number is not stuck in the author's head, it regenerates from the files. But read
the poster again. It says **4 points**, and the code says **3.41**. The reproduction
passed and the write-up still does not match the computation. That gap is the
first thing your red-team logs, and it is the reason "it ran" is never the finish
line.

> **A question that often comes up here:** *"The gap between 3.41 and 4 is tiny.
> Isn't flagging it just pedantic?"* On its own, a rounding stretch is minor. But
> it is a symptom worth naming, because it tells you the author reported a number
> the package does not produce. If they rounded 3.41 up to 4 without saying so,
> what else did they smooth over? The small mismatch earns you the right to ask the
> bigger questions coming next: which choices moved the number, and which
> assumptions hold it up.

### 🔍 Reading the Evidence: what a clean reproduction does and does not establish

You rebuilt the headline number from the package alone. In the cell below, write
three things. First: state exactly what your successful reproduction *does*
establish, in one sentence. Second: state what it does *not* establish, naming the
claims-vs-computation gap you already found (3.41 vs the poster's 4). Third: the
claim this module forbids is "it ran, so it's fine." Write the one sentence you
would say instead, to a peer who thinks a green check ends the audit.

### YOUR ANSWER HERE:

**What my successful reproduction DOES establish:**

**What it does NOT establish (name the 3.41-vs-4 gap):**

**What I'd say instead of 'it ran, so it's fine':**

---

## 3. Red-Team: Where the Number Quietly Does Not Hold

**Guiding question:** *once the number reproduces, which choices and assumptions
could move it, weaken it, or dissolve it?*

> *"Show me the claim next to the output that produced it. Then show me the choice
> you made that you did not mention, and tell me what happens to your number if I
> make the other reasonable choice instead."*
> — a **skeptical peer** reviewing the sample package with you

Three tools do most of a red-team's work, and this section runs each on the sample
package.

- **Claims-vs-computation agreement**: checking that every sentence the write-up
  asserts is backed by a number the code actually produces. Example: the poster
  says "4 points" but the code says 3.41, so those two disagree.
- **Alternative specification**: a different, equally defensible way to compute the
  same headline, to see whether the answer depends on a choice the author never
  disclosed. Example: weighting the data or not weighting it.
- **Hidden assumption**: a claim the analysis quietly relies on and never states,
  which the whole result would fall apart without. Example: treating every person
  as an independent observation when they were actually grouped by street.

> **A question that often comes up here:** *"The package matched its own number.
> Isn't hunting for problems just looking for trouble?"* Finding the trouble is the
> point, and it is a service, not an attack. A weakness you surface now, in
> private, becomes an honest caveat your peer can add. The same weakness left
> buried becomes the question that ends their defense. A red-team that finds
> nothing usually means the reviewer did not push, not that the package is
> flawless.

### Claims-vs-computation: line up the sentences against the outputs

The package's write-up makes a short list of claims. Run the cell to line each
claim up against what the code produces, so agreement and disagreement are on one
screen instead of in your memory.

In [ ]:
# The package's WRITTEN claims (from its poster), lined up against the COMPUTATION.
poster_claims = [
    "Canvassing raised turnout by 4 percentage points.",
    "Canvassing raised turnout. (no interval, no uncertainty stated)",
    "Reported as a single, settled number.",
]
computed = {
    "reproduced gap (points)": round(diff_pts, 2),
    "poster's stated gap (points)": 4.0,
    "gap between claim and code (points)": round(4.0 - diff_pts, 2),
}
for k, v in computed.items():
    print(f"  {k:<38}: {v}")

print("\nClaim-to-output trace:")
print(f"  1. '4 points'      -> code produces {diff_pts:.2f}. MISMATCH of "
      f"{4.0 - diff_pts:.2f} pts (the write-up rounds up, undisclosed).")
print("  2. 'raised turnout' -> a POINT number with NO uncertainty reported. The")
print("     package never states how firm 3.41 is, so a reader cannot tell signal")
print("     from noise. Missing uncertainty is a reporting gap, not a detail.")
print("  3. 'settled number' -> section 3.2 shows the number MOVES under a")
print("     defensible alternative choice the poster never mentioned.")

### ⚖️ Make a Design Choice: which specification is defensible?

*(Human-first: settle your own choice and its defense before you ask any AI.)*

The package data ships **weights**, a per-person number that corrects for who was
easier or harder to reach, so the sample better matches the target population. The
author computed the headline **without** using them, and never said so. You have
three ways to report the gap. **Choose one and defend it** in a short paragraph,
naming what each choice assumes and what it lets the claim say:

- **A.** The **unweighted** gap the author reported (3.41 points), and state that
  weights were available but not used.
- **B.** The **weighted** gap, and state that you switched because weights are what
  make the sample stand in for the population.
- **C.** Report **both**, and treat the spread between them as part of the finding.

In the cell below, pick A, B, or C, then defend the choice: what does your reported
number let the claim say, and where must that claim stop?

### YOUR ANSWER HERE:

**My chosen specification (A / B / C):**

**What each choice assumes, and why mine is defensible:**

**What my reported number lets the claim say, and where it must stop:**

---

**Deeper pass — run this on your own during the module.**
**Before you ask:** write your own one-sentence guess for how far the number moves
when you switch the weighting choice. Commit the size before the tool suggests one.

> 💡 **Gemini Prompt:** "Here is a cell that recomputes a study's headline turnout
> gap two ways, unweighted and weighted, from the same data: [paste the next cell].
> The author reported only the unweighted number and did not mention weights.
> Explain what weighting changes about who counts, and give me two competing
> readings of why the two numbers differ that do NOT assume one is simply correct.
> Then name what I would check in the data to tell your two readings apart."
>
> **After running, verify (counters *silent scope change*: 'the effect is 3.41' is
> a different claim than 'the effect is 3.41 under one undisclosed choice'):**
> - [ ] Confirm the two numbers Gemini discusses match the unweighted and weighted
>   gaps your cell printed, and the size of the spread between them in points.
> - [ ] Reject any sentence that treats one specification as automatically the
>   'true' one. The finding is that the choice moves the number and was not disclosed.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### 🛠️ Hands-On: the alternative specification (weighted vs unweighted)

Run the cell. It recomputes the exact same headline gap using the weights the
package shipped but never applied, and prints both numbers side by side.

In [ ]:
# ALTERNATIVE SPECIFICATION: the same gap, weighted with the shipped weights.
def weighted_mean(values, weights):
    """A weighted average: each person counts in proportion to their weight."""
    return np.average(values, weights=weights)

w_canv = weighted_mean(pkg.loc[pkg.treat == 1, y], pkg.loc[pkg.treat == 1, "weights"])
w_notc = weighted_mean(pkg.loc[pkg.treat == 0, y], pkg.loc[pkg.treat == 0, "weights"])
diff_w_pts = (w_canv - w_notc) * 100

print(f"  unweighted gap (what the author reported): {diff_pts:.2f} points")
print(f"  weighted gap   (weights the author skipped): {diff_w_pts:.2f} points")
print(f"  the number MOVES by {diff_pts - diff_w_pts:.2f} points on a choice the poster never named.")

In [ ]:
# Self-check: the specification choice moves the headline by a real amount.
assert abs(diff_w_pts - 2.83) < 0.05, "weighted gap drifted — investigate"
assert diff_pts - diff_w_pts > 0.3, "the two specifications should differ noticeably"
print(f"✓ Self-check passed: unweighted {diff_pts:.2f} vs weighted {diff_w_pts:.2f} "
      f"points.\n  A {diff_pts - diff_w_pts:.2f}-point swing hung on one undisclosed decision.")

**Reading the evidence.** Same data, same question, one defensible change: the
gap slides from **3.41** points unweighted to **2.83** points weighted. Neither is
a lie. The finding is that the headline depends on a choice the poster never
disclosed, and the honest report is the range, not the larger endpoint. You have
now moved the number by changing a *choice*. Next you push on an *assumption*, and
that one does more damage.

### The hidden assumption: were these really independent observations?

The package reports its gap as if every one of the 8,375 people were an independent
observation. They were not. Canvassing happened **by street**, so people on the
same street share a canvasser, a neighborhood, and a turnout habit. When grouped
data are treated as independent, the reported certainty is borrowed, not earned.

A **cluster** is a group of observations that are more alike inside the group than
across groups, so they carry less independent information than their raw count
suggests. Example: 20 neighbors on one canvassed street are not 20 independent
facts about canvassing. The honest way to size the uncertainty is to resample whole
**streets**, not individual people, and see how much the gap wobbles.

### 🔮 Pause & Predict

The next cell builds the uncertainty two ways. First the **naive** way, treating
all 8,375 people as independent. Then the **cluster-aware** way, resampling whole
streets. Before you run it, commit: will the cluster-aware interval be **narrower**,
**about the same**, or **wider** than the naive one? And here is the sharp question:
the naive interval sits clearly above zero. When you respect the street grouping,
will the interval still stay above zero, or reach down to it? Write one sentence of
reasoning.

### YOUR ANSWER HERE:

**Narrower / same / wider, and why:**

**Will the honest interval still exclude zero?**

---

**Deeper pass — run this on your own during the module.**
**Before you ask:** write your own one sentence naming what a package's AI-written
limitations paragraph would have to mention for you to trust it here. Commit that
before you read what the tool says.

> 💡 **Gemini Prompt:** "A peer's package includes this AI-generated limitations
> paragraph: 'This study has limitations. The sample may not represent all voters,
> the turnout measure is self-reported in places, and future work should use a
> larger sample.' The analysis treats 8,375 canvassed individuals as independent,
> though canvassing was assigned by street. Acting as a methods reviewer, tell me
> the single most important limitation this paragraph LEFT OUT, and why a
> street-clustered design makes the reported certainty too high."
>
> **After running, verify (counters *plausible-but-wrong-method*: an
> independent-observations assumption on clustered data quietly shrinks every
> interval):**
> - [ ] Confirm the omission it names is the **clustering / non-independence** one,
>   not a generic 'bigger sample' line. Check it against the naive-vs-cluster
>   interval your next cell prints.
> - [ ] Do not accept a fluent limitations paragraph as complete. Confirm the
>   cluster-aware interval yourself before you believe the certainty was overstated.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### 🛠️ Hands-On: the honest interval respects the clustering

Run the cell. It computes the naive 95% interval (people as independent), then the
cluster-aware interval (resampling whole streets), and prints both. Read the two
lower bounds closely.

In [ ]:
# NAIVE uncertainty: treat all people as independent.
n1, n0 = len(canvassed), len(not_canv)
naive_se = np.sqrt(canvassed.var(ddof=1) / n1 + not_canv.var(ddof=1) / n0)
naive_lo, naive_hi = (diff - 1.96 * naive_se) * 100, (diff + 1.96 * naive_se) * 100

# CLUSTER-AWARE uncertainty: resample whole STREETS, not individuals (seed 464).
# Precompute each street's treated/control vote sums and counts (version-robust).
tmp = pd.DataFrame({
    "street": pkg["street"],
    "yt": pkg[y] * (pkg.treat == 1), "nt": (pkg.treat == 1).astype(int),
    "yc": pkg[y] * (pkg.treat == 0), "nc": (pkg.treat == 0).astype(int),
})
per = tmp.groupby("street")[["yt", "nt", "yc", "nc"]].sum()
yt, nt = per["yt"].to_numpy(float), per["nt"].to_numpy(float)
yc, nc = per["yc"].to_numpy(float), per["nc"].to_numpy(float)
K = len(per)

boot = np.empty(400)
for b in range(400):
    idx = rng.integers(0, K, size=K)                 # resample whole streets
    boot[b] = yt[idx].sum() / nt[idx].sum() - yc[idx].sum() / nc[idx].sum()
cluster_se = boot.std()
cluster_lo, cluster_hi = (diff - 1.96 * cluster_se) * 100, (diff + 1.96 * cluster_se) * 100

print(f"  streets (clusters): {K}")
print(f"  NAIVE   95% interval: [{naive_lo:5.2f}, {naive_hi:5.2f}] points  (SE {naive_se:.4f})")
print(f"  CLUSTER 95% interval: [{cluster_lo:5.2f}, {cluster_hi:5.2f}] points  (SE {cluster_se:.4f})")
print(f"\n  Respecting the streets makes the interval about {cluster_se / naive_se:.1f} times wider.")
print(f"  The naive interval sits above zero; the cluster interval reaches down to about zero.")

# Plot the two intervals so the reach-to-zero is visible. Each interval is drawn
# in its own hue AND labeled directly, so nothing is encoded by color alone.
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.errorbar(diff_pts, 1.0, xerr=1.96 * naive_se * 100, fmt="o", color="#DD8452",
            capsize=6, elinewidth=2.5, label="naive (people independent)")
ax.errorbar(diff_pts, 0.0, xerr=1.96 * cluster_se * 100, fmt="s", color="#4C72B0",
            capsize=6, elinewidth=2.5, label="cluster-aware (whole streets)")
ax.axvline(0, color="gray", linestyle=":", linewidth=1.5, label="no effect (zero)")
ax.set_yticks([0.0, 1.0])
ax.set_yticklabels(["cluster-aware", "naive"])
ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("Turnout gap (percentage points), 95% interval")
ax.set_title("Same estimate, honest uncertainty: the cluster-aware interval reaches zero")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: honoring the clustering widens the interval toward zero.
assert cluster_se > 1.3 * naive_se, "cluster SE should be clearly larger than naive"
assert naive_lo > 0, "the naive interval should sit above zero"
assert cluster_lo < naive_lo, "the honest interval must reach lower than the naive one"
print(f"✓ Self-check passed: the naive lower bound is {naive_lo:.2f} (above zero); the")
print(f"  cluster-aware lower bound is {cluster_lo:.2f}, roughly touching zero. Same data,")
print(f"  honest uncertainty: the 'clearly positive' reading weakens once streets count.")

**Reading the evidence.** This is the weakness with teeth. The naive interval,
which treats every person as independent, runs from about **1.0 to 5.8 points** and
sits safely above zero, so it reads as a clear effect. Resample whole **streets**,
which is what the design actually was, and the interval widens by roughly **1.4
times** and its lower bound drops to **about zero**. The point estimate did not
change. The honest uncertainty around it did, and enough to weaken the "clearly
positive" reading. This is an **AI-related weakness** worth naming: the package's
AI-written limitations paragraph looked thorough, listing sample size and
self-report, yet it never mentioned the clustering that most threatens the claim.

That polish-over-substance failure has a name. The **illusion of completeness** is
an output that looks thorough, with many sections and a confident tone, while
quietly omitting the one thing that matters most. Example: a limitations paragraph
that covers three minor caveats and skips the clustering that could sink the
result. The cure is not to admire the list. It is to ask what the list left out,
then check it yourself, which is exactly what you just did.

### 🔬 Interrogate the Output

You now hold three findings against the package: a claims-vs-computation mismatch
(3.41 vs 4), a specification that moves the number (3.41 vs 2.83), and a clustering
assumption that widens the interval to about zero. Before you rank them, interrogate
your **own** audit against the printed numbers, using four checks. Answer each in
the cell below.

- **Claims:** does each of your three findings point to a real number your cells
  printed, not to an impression? Quote the number beside each.
- **Assumptions:** the clustering critique assumes people on a street are not
  independent. Is that assumption yours to defend, and can you say in one sentence
  why the design justifies it?
- **Missing information:** which weakness might you still be missing? A red-team
  that stops at three has not proven there is no fourth.
- **Overstatements:** does any of your findings sound more damning than the number
  supports? The specification swing is about 0.6 points, real but modest. Flag any word
  that oversells it.

And the rule that governs the whole module: **a clean run is not a correct claim.**
A cell can execute perfectly and still compute a number the write-up misreports.
Verify the *number and its match to the claim*, not just the green check.

### YOUR ANSWER HERE:

**Claims (each finding + the exact number beside it):**

**Assumptions (why the street-clustering critique is justified):**

**Missing information (the weakness I might still be missing):**

**Overstatements (any word that oversells a finding):**

---

## 4. The Required GenAI Studio Pass, Verified by You

**Guiding question:** *how do you use a second AI reader without letting two tools
agree their way into a wrong answer?*

M13 requires one pass through Purdue's **GenAI Studio Reproducibility Auditor**, a
course-configured role that reads a package the way a cold replicator would and
returns a **reproduction gap list** and a **claim-to-output trace**. Open the
course group's **HONR464 — Reproducibility Auditor** workspace, paste the cleared
contents of your assigned package, and read what it returns. The role proposes. You
confirm. It cannot run code, so every gap it names is a suspect you test yourself,
not a verdict.

One trap deserves its own name. **Correlated errors across tools** means two AI
readers make the *same* mistake, so their agreement feels like confirmation when it
is just one flaw echoed twice. Example: Gemini and the Auditor both miss the street
clustering and both call the package sound. If two AI sources agree, that is only
reassuring when they could fail in different ways. The independent check that breaks
the tie is your own run, which is why the interval you computed above outranks
anything either tool says.

> **A question that often comes up here:** *"If the Auditor and Gemini both say the
> package reproduces, isn't that two votes for yes?"* No, it is one vote counted
> twice. Neither tool executed the code. A package reproduces when *you* run it and
> the number comes back, full stop. Two models agreeing on a reading they both did
> without running anything is exactly the correlated-error trap. Your run is the
> only evidence that settles it.

### 🔁 Modify the Prompt

The lineage-trace prompt in section 2 was written for the sample package. Now retarget
it at **your assigned peer package**. First, in one or two sentences, write your own
prediction of where that package is weakest, before you ask any AI. Then adapt the
prompt so it names your package's data, its run order, and the one claim you most
doubt.

> **Base prompt (the lineage trace, from section 2):** "Here is everything a peer
> package shipped: [paste]. List every input or step a stranger needs and might not
> find, in a table with the item, whether the package supplies it, and the cell to
> check. Then name the one link where the written claim is most likely to have
> detached from an actual output."

Rewrite it so it names *your* package's headline number, *your* suspected weak
link, and asks the tool to rank its gap list by threat to the claim rather than by
ease of fixing. In the cell below: paste your weakness prediction, paste your
adapted prompt, predict the single gap the tool will rank first, then run it and
note whether your prediction held.

### YOUR ANSWER HERE:

**Where I predict my assigned package is weakest (AI did not draft this):**

**My adapted lineage-and-threat prompt (my data, my run order, my doubted claim):**

**The gap I predict the tool will rank first:**

**What the tool actually ranked first, and how I checked it against my own run:**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini and the Auditor for this one. No AI, no role, no notebook search. The
**red-team verdict** is a never-delegate decision: which weakness most threatens the
claim, and how you rank the rest, rests on your own run and your own judgment. In
the cell below, write, in your own words:

1. The **headline claim** of your assigned package, as its write-up states it.
2. Your **three ranked weaknesses**, most threatening first. For each, name the
   number or output that exposed it, and say in one phrase whether it *moves* the
   estimate, *widens* its uncertainty, or *breaks* the claim outright.
3. The single weakness you would put at the top of the board, and the one check
   that would confirm or dissolve it.

If you cannot yet rank them because a check is still unrun, that is a finding, not
a failure. It names the run you still owe before Sunday.

### YOUR ANSWER HERE:

**1. The package's headline claim (as written):**

**2. My three ranked weaknesses (most threatening first, with the number behind each):**

**3. The top weakness for the board, and the check that would settle it:**

---

## 5. Prioritized Recommendations: Rank by Threat, Not by Ease

**Guiding question:** *of everything you found, what should your peer fix first?*

A red-team that lists ten problems in the order it happened to find them is not
finished. The final move is to produce **prioritized recommendations**, also called
**triaged recommendations**: your findings ordered by how much each one threatens
the headline claim, not by how easy each is to fix. Example: a clustering error that
could sink the result outranks a typo in a variable name, even though the typo is
faster to repair.

Ranked for the sample package, the order is clear. The **clustering assumption**
comes first, because respecting it pushes the interval down to about zero and could
dissolve the "clearly positive" claim. The **undisclosed specification** comes
second, because it moves the number by about 0.6 points and hides that a choice was made.
The **claims-vs-computation mismatch** (3.41 reported as 4, with no uncertainty)
comes third: real, and a symptom of loose reporting, but the smallest threat to
whether an effect exists. Ease would have flipped this order. Threat is the only
ranking that helps your peer.

> **A question that often comes up here:** *"Shouldn't I lead with the easy fixes so
> my peer gets a quick win?"* Not in a red-team. Ease is the author's concern when
> they sit down to revise. Your job is to tell them where the claim is in the most
> danger, so they spend their limited time on what could actually change the
> verdict. Lead with the threat. Let them decide the repair order.

### 📝 Practice: three ways a clean run still hides a broken claim

*(Human-first: write and rank all three yourself before any AI. Catching these by
eye is the graded skill.)*

A package can pass restart-and-run-all and still carry a broken claim. In the cell
below, name **three distinct ways** that happens, drawing on what you found in the
sample package or on new ones. For each, give a one-line example. Then **rank your
three by threat to the claim**, most dangerous first, and justify the ranking in one
sentence.

> 💡 **Gemini Prompt:** "Here are three ways I claim a clean-running package can still
> hide a broken claim, with my ranking by threat: [paste your three, ranked]. As a
> skeptical methods reviewer, tell me whether my ranking actually orders by threat to
> the headline claim or slips into ordering by how easy each is to fix. Then name one
> more way a clean run hides a broken claim that my three missed."
>
> **After running, verify (counters *illusion of completeness*: three tidy items can
> feel exhaustive while missing the failure that matters):**
> - [ ] Check the tool's extra item against your own list. If it names a real fourth
>   way you missed, your "three" was not complete, so add it.
> - [ ] Confirm your ranking still orders by threat, not ease, after reading its
>   critique. Reject any reordering that moves the easy fix to the top.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### YOUR ANSWER HERE:

**Way 1 (+ one-line example):**

**Way 2 (+ one-line example):**

**Way 3 (+ one-line example):**

**My ranking by threat (most dangerous first) and why:**

---

### 🎯 Project Transfer: the spine of your M13 red-team report

*(Human-first: draft every piece from your own assigned package. AI may critique it
after, but the report you submit Sunday has to be reasoned by you.)*

This is where the module becomes your **M13 Replication and Red-Team Report**. In
the cell below, draft the report's spine for the package you were assigned:

1. **Reproduction log:** did restart-and-run-all succeed? What headline number did
   you get from the package alone, and did it match the write-up? Log every break
   with the exact cell.
2. **Claim-to-output trace:** each written claim beside the output that supports it,
   with every mismatch flagged.
3. **One alternative specification and one hidden assumption:** what you changed or
   questioned, and how the number moved, widened, or held.
4. **The GenAI Studio pass, verified:** what the Reproducibility Auditor flagged,
   and which of its flags survived your own run.
5. **Prioritized recommendations:** your findings ranked by threat to the claim, and
   the single most-threatening weakness you will post to the async board with the
   check that would settle it.

Write these five so a reader who has never seen the package could locate the exact
edge of its evidence.

### YOUR ANSWER HERE:

**1. Reproduction log (run result + headline number + matches?):**

**2. Claim-to-output trace (claims vs outputs, mismatches flagged):**

**3. One alternative specification + one hidden assumption (what moved):**

**4. GenAI Studio pass, verified (what survived my own run):**

**5. Prioritized recommendations + my most-threatening weakness for the board:**

---

### 📒 AI Research Ledger

Log every AI use from this module in the ledger. Two worked rows are filled in as
models: one for the reproduction walkthrough, one for the required GenAI Studio pass.
They model the discipline this module teaches: **the AI proposes suspects, your run
confirms.** Add a row for each prompt and each role pass you actually ran. The ledger
is a graded habit, not paperwork. It is how you show your audit was verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Walk through the reproduction cell and give an independent confirm | Gemini | "Explain this diff-in-means cell line by line and confirm the headline value; give one independent way to check it." | Correctly identified the 3.41-point gap; independent recompute matched | Accepted the walkthrough after my own recompute agreed | Reran the group means by hand from the printout; confirmed 50.66 vs 47.25 | The tool cannot run code, so I trust only my executed number | *(your name)* |
| Audit the package for reproduction gaps | GenAI Studio Reproducibility Auditor | Pasted the cleared package contents for the gap list and claim-to-output trace | Flagged the undisclosed weighting; missed the street clustering | Kept the weighting flag; added the clustering weakness the role missed | Ran the cluster-aware interval myself; it, not the role, exposed the biggest threat | Gemini and the role could miss the same runtime blocker (correlated error) | *(your name)* |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #13 — write, in your own words:

1. **The claim I can defend:** one bounded sentence about your assigned package,
   such as "I reproduced the headline number, and its most threatening weakness is
   X." Put your name on it.
2. **Its boundary:** what your audit does NOT establish. A reproduction confirms the
   number regenerates, not that the science is right, and a red-team names risks it
   does not by itself resolve.
3. **My uncertainty and limitations:** one sentence naming what you could not fully
   check, such as a step that would not run or a claim you could not trace.
4. **AI check:** what you delegated to Gemini and the GenAI Studio role, whether you
   **accepted, changed, or rejected** what each returned, and the specific run that
   decided it.

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (what reproduction and red-team do NOT establish):**

**3. My uncertainty and limitations:**

**4. AI check (Gemini + GenAI Studio: accepted, changed, or rejected, and how verified):**

---

## 6. Wrap-Up

In one self-paced sitting you took a stranger's finished package and held it to the
standard you will be held to. You ran restart-and-run-all and reproduced the headline
number from the files alone, then refused to stop at "it ran." You traced the
data-and-code lineage, lined each written claim against the output behind it, and
found the write-up's 4 points was a computed 3.41 with no uncertainty. You moved the
number with an undisclosed specification, and you widened its interval to about zero
by respecting the street clustering the AI limitations paragraph forgot. You ran the
required GenAI Studio pass, caught the correlated-error trap, and verified the real
threat with your own run. Then you ranked your findings by threat, not by ease.

> **"A clean run certifies that code executes. It never certifies that the sentence
> on the poster is true. Reproduction earns you a trustworthy number; the red-team
> is where you find out what that number can and cannot hold. 'It ran, so it's fine'
> is the one verdict this module exists to refuse."**

When class resumes, the lens turns inward. The next unit begins your **research
note** (nb14, M14, posted today to read over the break), where the package you audit
is your own, and every claim you write has to survive the exact trace you just ran
on a peer. Submit **M13 by Sunday night**, post your most-threatening weakness to the
board, and reply to one classmate. Then the rest of the break is genuinely yours.
Happy Thanksgiving.

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *project/final_dossier/reproducibility_audit_protocol.md | cold-reproduction, the five package sins, and the claim-to-output standard | reframed as a self-paced reproduce-then-red-team module | adapted*
- *genai_studio/roles/reproducibility_auditor.md (M13 required touchpoint) | the reproduction-gap-list + claim-to-output-trace role, and its correlated-error warning | referenced as the required GenAI Studio pass | referenced*
- *foos_etal.csv | rdss package data | a real UK get-out-the-vote field experiment used as the sample package to reproduce (headline gap, weighted specification, street-clustered interval) | adapted (the file stands in as an anonymized peer package)*
- *ai_resources/ai_error_taxonomy.md | the named failures each verify checklist counters (confident fabrication, silent scope change, illusion of completeness, plausible-but-wrong-method, correlated errors) | referenced*
- *fresh | the lineage diagram + the prioritized-by-threat recommendation frame | newly-constructed-from-source-concept*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- No new reading: this is a self-contained async module, and the
  reproducibility-audit protocol stands in for the chapter. As a companion,
  Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 21 and ch. 22 (the Redesign section on realization and
  reconciliation). Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>